# EXPLORATORY ANALYSIS (EDA)

### Crear conexión y cargar tablas

In [1]:
#Creamos la conexión a Snowflake con Snowpark
from snowflake.snowpark import Session
import os
from dotenv import load_dotenv

#Carga de variables 
load_dotenv("credentials.env")

#Parámetros de conexión a Snowflake
con = {
      "account": os.getenv("SNOWFLAKE_ACCOUNT"),
      "database": os.getenv("SNOWFLAKE_DATABASE"),
      "password": os.getenv("SNOWFLAKE_PASSWORD"),
      "role": "ACCOUNTADMIN",
      "schema": "TPCH_SF1",
      "user": os.getenv("SNOWFLAKE_USER"),
      "warehouse": os.getenv("SNOWFLAKE_WAREHOUSE")
}

#Creamos la sesión de Snowpark
session = Session.builder.configs(con).create()

##### Debug de la conexión

In [2]:
df_cus = session.table("TPCH_SF1.CUSTOMER")
df_cus.show()

----------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------
|"C_CUSTKEY"  |"C_NAME"            |"C_ADDRESS"                              |"C_NATIONKEY"  |"C_PHONE"        |"C_ACCTBAL"  |"C_MKTSEGMENT"  |"C_COMMENT"                                         |
----------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------
|60001        |Customer#000060001  |9Ii4zQn9cX                               |14             |24-678-784-9652  |9957.56      |HOUSEHOLD       |l theodolites boost slyly at the platelets: per...  |
|60002        |Customer#000060002  |ThGBMjDwKzkoOxhz                         |15             |25-782-500-8435  |742.46       |BUILDING        | beans. fluffily regular packages                   |
|60003        |

### Carga de librerías específicas para el EDA

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import statistics
import scipy.stats as stats
import random

# Configuración para mostrar todas las columnas del DataFrame
pd.set_option('display.max_columns', None)

#### Overview del esquema TPCH_SF1

In [4]:
df_tables = session.sql("SHOW TABLES IN SCHEMA TPCH_SF1")
df_tables.show()

-------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------
|"created_on"                      |"name"    |"database_name"        |"schema_name"  |"kind"  |"comment"                          |"cluster_by"  |"rows"   |"bytes"    |"owner"  |"retention_time"  |"automatic_clustering"  |"change_tracking"  |"search_optimization"  |"search_optimization_progress"  |"search_optimization_bytes"  |"is_external"  |"enable_schema_evolution"  |"owner_role_type"  |"is_event"  |"is_hybrid"  |"is_iceberg"  |"is_dynamic"  |"is_immutable"  |"is_interactive"  |
------------------------

#### Análisis general y calidad de datos de las tablas

###### CUSTOMER

In [5]:
cus = session.sql("SELECT * FROM TPCH_SF1.CUSTOMER")
cus.show()

#Creamos objeto pandas para poder explorar y visualizar los datos
df_cus = cus.to_pandas()
df_cus.head()

----------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------
|"C_CUSTKEY"  |"C_NAME"            |"C_ADDRESS"                              |"C_NATIONKEY"  |"C_PHONE"        |"C_ACCTBAL"  |"C_MKTSEGMENT"  |"C_COMMENT"                                         |
----------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------
|60001        |Customer#000060001  |9Ii4zQn9cX                               |14             |24-678-784-9652  |9957.56      |HOUSEHOLD       |l theodolites boost slyly at the platelets: per...  |
|60002        |Customer#000060002  |ThGBMjDwKzkoOxhz                         |15             |25-782-500-8435  |742.46       |BUILDING        | beans. fluffily regular packages                   |
|60003        |

,C_CUSTKEY,C_NAME,C_ADDRESS,C_NATIONKEY,C_PHONE,C_ACCTBAL,C_MKTSEGMENT,C_COMMENT
0,60001,Customer#000060001,9Ii4zQn9cX,14,24-678-784-9652,9957.56,HOUSEHOLD,l theodolites boost slyly at the platelets: pe...
1,60002,Customer#000060002,ThGBMjDwKzkoOxhz,15,25-782-500-8435,742.46,BUILDING,beans. fluffily regular packages
2,60003,Customer#000060003,"Ed hbPtTXMTAsgGhCr4HuTzK,Md2",16,26-859-847-7640,2526.92,BUILDING,fully pending deposits sleep quickly. blithely...
3,60004,Customer#000060004,"NivCT2RVaavl,yUnKwBjDyMvB42WayXCnky",10,20-573-674-7999,7975.22,AUTOMOBILE,furiously above the ironic packages. slyly br...
4,60005,Customer#000060005,"1F3KM3ccEXEtI, B22XmCMOWJMl",12,22-741-208-1316,2504.74,MACHINERY,express instructions sleep quickly. ironic bra...


In [6]:
# Información genérica de la tabla customer
df_cus.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 150000 entries, 0 to 149999
Data columns (total 8 columns):
 #   Column        Non-Null Count   Dtype  
---  ------        --------------   -----  
 0   C_CUSTKEY     150000 non-null  int32  
 1   C_NAME        150000 non-null  object 
 2   C_ADDRESS     150000 non-null  object 
 3   C_NATIONKEY   150000 non-null  int8   
 4   C_PHONE       150000 non-null  object 
 5   C_ACCTBAL     150000 non-null  float64
 6   C_MKTSEGMENT  150000 non-null  object 
 7   C_COMMENT     150000 non-null  object 
dtypes: float64(1), int32(1), int8(1), object(5)
memory usage: 7.6+ MB


In [7]:
# Estadísticas descriptivas de la tabla customer
df_cus.describe(include = "all").T

,count,unique,top,freq,mean,std,min,25%,50%,75%,max
C_CUSTKEY,150000.0,NaN,NaN,NaN,75000.5,43301.414527,1.0,37500.75,75000.5,112500.25,150000.0
C_NAME,150000,150000,Customer#000029984,1,NaN,NaN,NaN,NaN,NaN,NaN,NaN
C_ADDRESS,150000,150000,GyXgbJqMafzZrSlU25khYBTTK8XXp3 sfikh,1,NaN,NaN,NaN,NaN,NaN,NaN,NaN
C_NATIONKEY,150000.0,NaN,NaN,NaN,12.0067,7.208184,0.0,6.0,12.0,18.0,24.0
C_PHONE,150000,150000,10-384-618-1669,1,NaN,NaN,NaN,NaN,NaN,NaN,NaN
C_ACCTBAL,150000.0,NaN,NaN,NaN,4495.512332,3174.32237,-999.99,1757.62,4477.3,7246.315,9999.99
C_MKTSEGMENT,150000,5,HOUSEHOLD,30189,NaN,NaN,NaN,NaN,NaN,NaN,NaN
C_COMMENT,150000,149968,posits engage across the furiousl,2,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [8]:
# Identificación de valores nulos
df_cus.isna().sum().sort_values(ascending = False)

C_CUSTKEY       0
C_NAME          0
C_ADDRESS       0
C_NATIONKEY     0
C_PHONE         0
C_ACCTBAL       0
C_MKTSEGMENT    0
C_COMMENT       0
dtype: int64

In [9]:
# Identificación de resgistros duplicados
df_cus.duplicated().sum()

np.int64(0)

In [10]:
# Identificación de clientes únicos
df_cus.C_CUSTKEY.nunique()

150000

###### LINEITEM

In [11]:
lin = session.sql("SELECT * FROM TPCH_SF1.LINEITEM")
lin.show()

#Creamos objeto pandas para poder explorar y visualizar los datos
df_lin = lin.to_pandas()
df_lin.head()

---------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------
|"L_ORDERKEY"  |"L_PARTKEY"  |"L_SUPPKEY"  |"L_LINENUMBER"  |"L_QUANTITY"  |"L_EXTENDEDPRICE"  |"L_DISCOUNT"  |"L_TAX"  |"L_RETURNFLAG"  |"L_LINESTATUS"  |"L_SHIPDATE"  |"L_COMMITDATE"  |"L_RECEIPTDATE"  |"L_SHIPINSTRUCT"   |"L_SHIPMODE"  |"L_COMMENT"                                 |
---------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------
|2400001       |132304       |4818         |1               |10.00         |13363.00           |0.03          |0.02     |N               |O   

,L_ORDERKEY,L_PARTKEY,L_SUPPKEY,L_LINENUMBER,L_QUANTITY,L_EXTENDEDPRICE,L_DISCOUNT,L_TAX,L_RETURNFLAG,L_LINESTATUS,L_SHIPDATE,L_COMMITDATE,L_RECEIPTDATE,L_SHIPINSTRUCT,L_SHIPMODE,L_COMMENT
0,1200001,36262,8766,1,31.0,37146.06,0.06,0.03,R,F,1994-05-02,1994-03-30,1994-05-11,NONE,RAIL,"ular, express ins"
1,1200001,54065,1581,2,23.0,23438.38,0.00,0.03,A,F,1994-01-26,1994-03-18,1994-02-07,DELIVER IN PERSON,SHIP,coys after the slyly even idea
2,1200002,167367,7368,1,40.0,57374.40,0.01,0.00,N,O,1997-03-08,1997-03-01,1997-03-15,COLLECT COD,SHIP,accounts cajole beside the furiously e
3,1200002,131610,9150,2,29.0,47606.69,0.01,0.01,N,O,1997-02-13,1997-01-15,1997-03-14,TAKE BACK RETURN,RAIL,e fluffily fluffily regular pin
4,1200002,161851,9400,3,23.0,43995.55,0.00,0.04,N,O,1997-01-17,1997-01-12,1997-02-16,COLLECT COD,TRUCK,"nal platelets. regular, special Tiresias pl"


In [65]:
df_lin.columns.to_list()

['L_ORDERKEY',
 'L_PARTKEY',
 'L_SUPPKEY',
 'L_LINENUMBER',
 'L_QUANTITY',
 'L_EXTENDEDPRICE',
 'L_DISCOUNT',
 'L_TAX',
 'L_RETURNFLAG',
 'L_LINESTATUS',
 'L_SHIPDATE',
 'L_COMMITDATE',
 'L_RECEIPTDATE',
 'L_SHIPINSTRUCT',
 'L_SHIPMODE',
 'L_COMMENT']

In [12]:
# Información genérica de la tabla lineitem
df_lin.info(memory_usage='deep')

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 6001215 entries, 0 to 6001214
Data columns (total 16 columns):
 #   Column           Dtype  
---  ------           -----  
 0   L_ORDERKEY       int32  
 1   L_PARTKEY        int32  
 2   L_SUPPKEY        int16  
 3   L_LINENUMBER     int8   
 4   L_QUANTITY       float64
 5   L_EXTENDEDPRICE  float64
 6   L_DISCOUNT       float64
 7   L_TAX            float64
 8   L_RETURNFLAG     object 
 9   L_LINESTATUS     object 
 10  L_SHIPDATE       object 
 11  L_COMMITDATE     object 
 12  L_RECEIPTDATE    object 
 13  L_SHIPINSTRUCT   object 
 14  L_SHIPMODE       object 
 15  L_COMMENT        object 
dtypes: float64(4), int16(1), int32(2), int8(1), object(8)
memory usage: 2.8 GB


In [13]:
# Estadísticas descriptivas de la tabla customer
df_lin.describe(include = "all").T

,count,unique,top,freq,mean,std,min,25%,50%,75%,max
L_ORDERKEY,6001215.0,NaN,NaN,NaN,3000279.604205,1732187.87348,1.0,1500738.0,3000961.0,4500354.0,6000000.0
L_PARTKEY,6001215.0,NaN,NaN,NaN,100017.98933,57735.690827,1.0,50016.0,100000.0,150043.0,200000.0
L_SUPPKEY,6001215.0,NaN,NaN,NaN,5000.602606,2886.961999,1.0,2499.0,5001.0,7500.0,10000.0
L_LINENUMBER,6001215.0,NaN,NaN,NaN,3.000576,1.732431,1.0,2.0,3.0,4.0,7.0
L_QUANTITY,6001215.0,NaN,NaN,NaN,25.507967,14.426263,1.0,13.0,26.0,38.0,50.0
L_EXTENDEDPRICE,6001215.0,NaN,NaN,NaN,38255.138485,23300.438711,901.0,18739.1,36718.64,55158.94,104949.5
L_DISCOUNT,6001215.0,NaN,NaN,NaN,0.049999,0.03162,0.0,0.02,0.05,0.08,0.1
L_TAX,6001215.0,NaN,NaN,NaN,0.040014,0.025817,0.0,0.02,0.04,0.06,0.08
L_RETURNFLAG,6001215,3,N,3043852,NaN,NaN,NaN,NaN,NaN,NaN,NaN
L_LINESTATUS,6001215,2,O,3004998,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [14]:
# Identificación de valores nulos
df_lin.isna().sum().sort_values(ascending = False)

L_ORDERKEY         0
L_PARTKEY          0
L_SUPPKEY          0
L_LINENUMBER       0
L_QUANTITY         0
L_EXTENDEDPRICE    0
L_DISCOUNT         0
L_TAX              0
L_RETURNFLAG       0
L_LINESTATUS       0
L_SHIPDATE         0
L_COMMITDATE       0
L_RECEIPTDATE      0
L_SHIPINSTRUCT     0
L_SHIPMODE         0
L_COMMENT          0
dtype: int64

In [15]:
# Identificación de resgistros duplicados
df_lin.duplicated().sum()

np.int64(0)

In [16]:
# Identificación de clave única y de la duplicidad de producto-pedido en algún registro
df_lin.groupby(["L_ORDERKEY", "L_PARTKEY"]).size().sum()

np.int64(6001215)

###### NATION

In [17]:
nat = session.sql("SELECT * FROM TPCH_SF1.NATION")
nat.show()

#Creamos objeto pandas para poder explorar y visualizar los datos
df_nat = nat.to_pandas()
df_nat.head()

--------------------------------------------------------------------------------------------------
|"N_NATIONKEY"  |"N_NAME"   |"N_REGIONKEY"  |"N_COMMENT"                                         |
--------------------------------------------------------------------------------------------------
|0              |ALGERIA    |0              | haggle. carefully final deposits detect slyly ...  |
|1              |ARGENTINA  |1              |al foxes promise slyly according to the regular...  |
|2              |BRAZIL     |1              |y alongside of the pending deposits. carefully ...  |
|3              |CANADA     |1              |eas hang ironic, silent packages. slyly regular...  |
|4              |EGYPT      |4              |y above the carefully unusual theodolites. fina...  |
|5              |ETHIOPIA   |0              |ven packages wake quickly. regu                     |
|6              |FRANCE     |3              |refully final requests. regular, ironi              |
|7        

,N_NATIONKEY,N_NAME,N_REGIONKEY,N_COMMENT
0,0,ALGERIA,0,haggle. carefully final deposits detect slyly...
1,1,ARGENTINA,1,al foxes promise slyly according to the regula...
2,2,BRAZIL,1,y alongside of the pending deposits. carefully...
3,3,CANADA,1,"eas hang ironic, silent packages. slyly regula..."
4,4,EGYPT,4,y above the carefully unusual theodolites. fin...


In [18]:
# Información genérica de la tabla lineitem
df_nat.info(memory_usage='deep')

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 25 entries, 0 to 24
Data columns (total 4 columns):
 #   Column       Non-Null Count  Dtype 
---  ------       --------------  ----- 
 0   N_NATIONKEY  25 non-null     int8  
 1   N_NAME       25 non-null     object
 2   N_REGIONKEY  25 non-null     int8  
 3   N_COMMENT    25 non-null     object
dtypes: int8(2), object(2)
memory usage: 4.9 KB


In [19]:
# Estadísticas descriptivas de la tabla customer
df_nat.describe(include = "all").T

,count,unique,top,freq,mean,std,min,25%,50%,75%,max
N_NATIONKEY,25.0,NaN,NaN,NaN,12.0,7.359801,0.0,6.0,12.0,18.0,24.0
N_NAME,25,25,ALGERIA,1,NaN,NaN,NaN,NaN,NaN,NaN,NaN
N_REGIONKEY,25.0,NaN,NaN,NaN,2.0,1.443376,0.0,1.0,2.0,3.0,4.0
N_COMMENT,25,25,haggle. carefully final deposits detect slyly...,1,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [20]:
# Identificación de valores nulos
df_nat.isna().sum().sort_values(ascending = False)

N_NATIONKEY    0
N_NAME         0
N_REGIONKEY    0
N_COMMENT      0
dtype: int64

In [21]:
# Identificación de resgistros duplicados
df_nat.duplicated().sum()

np.int64(0)

In [22]:
# Identificación de países únicos
df_nat.N_NATIONKEY.nunique()

25

###### ORDERS

In [23]:
ord= session.sql("SELECT * FROM TPCH_SF1.ORDERS")
ord.show()

#Creamos objeto pandas para poder explorar y visualizar los datos
df_ord = ord.to_pandas()
df_ord.head()

-----------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------
|"O_ORDERKEY"  |"O_CUSTKEY"  |"O_ORDERSTATUS"  |"O_TOTALPRICE"  |"O_ORDERDATE"  |"O_ORDERPRIORITY"  |"O_CLERK"        |"O_SHIPPRIORITY"  |"O_COMMENT"                                         |
-----------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------
|3000001       |145618       |F                |30175.88        |1992-12-17     |4-NOT SPECIFIED    |Clerk#000000141  |0                 |l packages. furiously careful instructions grow...  |
|3000002       |1481         |O                |297999.63       |1995-07-28     |1-URGENT           |Clerk#000000547  |0                 |carefully unusual dependencie                       |
|3000003       |127432       |O         

,O_ORDERKEY,O_CUSTKEY,O_ORDERSTATUS,O_TOTALPRICE,O_ORDERDATE,O_ORDERPRIORITY,O_CLERK,O_SHIPPRIORITY,O_COMMENT
0,1800001,60685,F,314159.07,1994-10-03,1-URGENT,Clerk#000000306,0,. ironic packages wake
1,1800002,51488,F,5535.79,1993-07-08,3-MEDIUM,Clerk#000000897,0,deposits affix carefully. excuses boost fluffil
2,1800003,142558,F,193712.88,1992-01-29,4-NOT SPECIFIED,Clerk#000000137,0,ly regular accounts run carefully slyly final ...
3,1800004,18287,O,75527.58,1997-09-19,2-HIGH,Clerk#000000481,0,carefully furiously regular theodolites. iron...
4,1800005,139264,P,132162.89,1995-03-24,3-MEDIUM,Clerk#000000471,0,xpress deposits. regula


In [24]:
# Información genérica de la tabla lineitem
df_ord.info(memory_usage='deep')

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1500000 entries, 0 to 1499999
Data columns (total 9 columns):
 #   Column           Non-Null Count    Dtype  
---  ------           --------------    -----  
 0   O_ORDERKEY       1500000 non-null  int32  
 1   O_CUSTKEY        1500000 non-null  int32  
 2   O_ORDERSTATUS    1500000 non-null  object 
 3   O_TOTALPRICE     1500000 non-null  float64
 4   O_ORDERDATE      1500000 non-null  object 
 5   O_ORDERPRIORITY  1500000 non-null  object 
 6   O_CLERK          1500000 non-null  object 
 7   O_SHIPPRIORITY   1500000 non-null  int8   
 8   O_COMMENT        1500000 non-null  object 
dtypes: float64(1), int32(2), int8(1), object(5)
memory usage: 512.0 MB


In [25]:
# Estadísticas descriptivas de la tabla customer
df_ord.describe(include = "all").T

,count,unique,top,freq,mean,std,min,25%,50%,75%,max
O_ORDERKEY,1500000.0,NaN,NaN,NaN,2999991.5,1732051.38492,1.0,1500000.75,3000000.5,4500000.25,6000000.0
O_CUSTKEY,1500000.0,NaN,NaN,NaN,75006.040575,43304.489008,1.0,37490.0,74990.0,112535.0,149999.0
O_ORDERSTATUS,1500000,3,O,732044,NaN,NaN,NaN,NaN,NaN,NaN,NaN
O_TOTALPRICE,1500000.0,NaN,NaN,NaN,151219.537632,88621.431364,857.71,77894.7475,144409.04,215500.225,555285.16
O_ORDERDATE,1500000,2406,1995-01-13,702,NaN,NaN,NaN,NaN,NaN,NaN,NaN
O_ORDERPRIORITY,1500000,5,5-LOW,300589,NaN,NaN,NaN,NaN,NaN,NaN,NaN
O_CLERK,1500000,1000,Clerk#000000542,1618,NaN,NaN,NaN,NaN,NaN,NaN,NaN
O_SHIPPRIORITY,1500000.0,NaN,NaN,NaN,0.0,0.0,0.0,0.0,0.0,0.0,0.0
O_COMMENT,1500000,1482071,carefully regular,17,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [26]:
df_ord.O_SHIPPRIORITY.unique()

array([0], dtype=int8)

In [27]:
# Identificación de valores nulos
df_ord.isna().sum().sort_values(ascending = False)

O_ORDERKEY         0
O_CUSTKEY          0
O_ORDERSTATUS      0
O_TOTALPRICE       0
O_ORDERDATE        0
O_ORDERPRIORITY    0
O_CLERK            0
O_SHIPPRIORITY     0
O_COMMENT          0
dtype: int64

In [28]:
# Identificación de resgistros duplicados
df_ord.duplicated().sum()

np.int64(0)

In [29]:
# Identificación de pedidos únicos
df_ord.O_ORDERKEY.nunique()

1500000

###### PART

In [30]:
part= session.sql("SELECT * FROM TPCH_SF1.PART")
part.show()

#Creamos objeto pandas para poder explorar y visualizar los datos
df_part = part.to_pandas()
df_part.head()

------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------
|"P_PARTKEY"  |"P_NAME"                               |"P_MFGR"        |"P_BRAND"  |"P_TYPE"                 |"P_SIZE"  |"P_CONTAINER"  |"P_RETAILPRICE"  |"P_COMMENT"             |
------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------
|40001        |cream steel purple royal goldenrod     |Manufacturer#3  |Brand#34   |MEDIUM BURNISHED COPPER  |27        |JUMBO PKG      |941.00           |lly slow depende        |
|40002        |almond cyan forest midnight khaki      |Manufacturer#3  |Brand#34   |ECONOMY POLISHED TIN     |32        |WRAP BOX       |942.00           |ng to                   |
|40003        |drab thistle navajo light purple       |Manufacturer#5  |Brand#55   |SMALL ANODI

,P_PARTKEY,P_NAME,P_MFGR,P_BRAND,P_TYPE,P_SIZE,P_CONTAINER,P_RETAILPRICE,P_COMMENT
0,120001,yellow hot rose blue green,Manufacturer#1,Brand#15,STANDARD ANODIZED STEEL,38,SM BAG,1021.0,he unusual requests.
1,120002,yellow pale blanched gainsboro moccasin,Manufacturer#4,Brand#45,STANDARD PLATED BRASS,22,WRAP CAN,1022.0,e furiously even ex
2,120003,metallic rosy gainsboro dark spring,Manufacturer#4,Brand#45,PROMO ANODIZED NICKEL,48,LG DRUM,1023.0,ounts. c
3,120004,purple cream puff royal chocolate,Manufacturer#2,Brand#24,SMALL ANODIZED TIN,41,MED BAG,1024.0,ly final
4,120005,turquoise floral papaya steel blanched,Manufacturer#2,Brand#25,SMALL BURNISHED COPPER,35,SM DRUM,1025.0,s sleep. slyly final d


In [31]:
# Información genérica de la tabla part
df_part.info(memory_usage='deep')

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 200000 entries, 0 to 199999
Data columns (total 9 columns):
 #   Column         Non-Null Count   Dtype  
---  ------         --------------   -----  
 0   P_PARTKEY      200000 non-null  int32  
 1   P_NAME         200000 non-null  object 
 2   P_MFGR         200000 non-null  object 
 3   P_BRAND        200000 non-null  object 
 4   P_TYPE         200000 non-null  object 
 5   P_SIZE         200000 non-null  int8   
 6   P_CONTAINER    200000 non-null  object 
 7   P_RETAILPRICE  200000 non-null  float64
 8   P_COMMENT      200000 non-null  object 
dtypes: float64(1), int32(1), int8(1), object(6)
memory usage: 86.1 MB


In [32]:
# Estadísticas descriptivas de la tabla part
df_part.describe(include = "all").T

,count,unique,top,freq,mean,std,min,25%,50%,75%,max
P_PARTKEY,200000.0,NaN,NaN,NaN,100000.5,57735.171256,1.0,50000.75,100000.5,150000.25,200000.0
P_NAME,200000,199997,azure lime rosy peru powder,2,NaN,NaN,NaN,NaN,NaN,NaN,NaN
P_MFGR,200000,5,Manufacturer#3,40304,NaN,NaN,NaN,NaN,NaN,NaN,NaN
P_BRAND,200000,25,Brand#35,8233,NaN,NaN,NaN,NaN,NaN,NaN,NaN
P_TYPE,200000,150,ECONOMY ANODIZED STEEL,1451,NaN,NaN,NaN,NaN,NaN,NaN,NaN
P_SIZE,200000.0,NaN,NaN,NaN,25.427105,14.441559,1.0,13.0,25.0,38.0,50.0
P_CONTAINER,200000,40,SM DRUM,5153,NaN,NaN,NaN,NaN,NaN,NaN,NaN
P_RETAILPRICE,200000.0,NaN,NaN,NaN,1499.496,294.673834,901.0,1249.2475,1499.495,1749.7425,2098.99
P_COMMENT,200000,131753,the,123,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [33]:
# Identificación de valores nulos
df_part.isna().sum().sort_values(ascending = False)

P_PARTKEY        0
P_NAME           0
P_MFGR           0
P_BRAND          0
P_TYPE           0
P_SIZE           0
P_CONTAINER      0
P_RETAILPRICE    0
P_COMMENT        0
dtype: int64

In [34]:
# Identificación de resgistros duplicados
df_ord.duplicated().sum()

np.int64(0)

In [41]:
# Identificación de productos únicos
df_part.P_PARTKEY.value_counts().sort_values(ascending = False)

P_PARTKEY
120000    1
179986    1
179987    1
179988    1
179989    1
         ..
119995    1
119996    1
119997    1
119998    1
119999    1
Name: count, Length: 200000, dtype: int64

###### PARTSUPP

In [43]:
parts= session.sql("SELECT * FROM TPCH_SF1.PARTSUPP")
parts.show()

#Creamos objeto pandas para poder explorar y visualizar los datos
df_parts = parts.to_pandas()
df_parts.head()

----------------------------------------------------------------------------------------------------------------------
|"PS_PARTKEY"  |"PS_SUPPKEY"  |"PS_AVAILQTY"  |"PS_SUPPLYCOST"  |"PS_COMMENT"                                        |
----------------------------------------------------------------------------------------------------------------------
|60001         |2             |1770           |535.64           |inal instructions. express accounts wake fluffi...  |
|60001         |2508          |1294           |693.16           |iously express packages haggle. quickly special...  |
|60001         |5014          |6842           |824.77           |ecial, express foxes haggle blithely slyly regu...  |
|60001         |7520          |2607           |979.30           | pinto beans haggle quickly near the final depo...  |
|60002         |3             |6686           |746.49           |fully furiously final packages. carefully speci...  |
|60002         |2509          |1315           |9

,PS_PARTKEY,PS_SUPPKEY,PS_AVAILQTY,PS_SUPPLYCOST,PS_COMMENT
0,40001,2,9531,683.63,structions. bold packages use furiously agains...
1,40001,2506,3331,372.02,"usy deposits; bold, express platelets are thin..."
2,40001,5010,4324,21.33,iously regular accounts; requests wake furious...
3,40001,7514,9319,95.15,ilent asymptotes. carefully bold requests afte...
4,40002,3,953,17.33,posits cajole fluffily. packages serve. fluffi...


In [45]:
# Información genérica de la tabla partsupp
df_parts.info(memory_usage='deep')

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 800000 entries, 0 to 799999
Data columns (total 5 columns):
 #   Column         Non-Null Count   Dtype  
---  ------         --------------   -----  
 0   PS_PARTKEY     800000 non-null  int32  
 1   PS_SUPPKEY     800000 non-null  int16  
 2   PS_AVAILQTY    800000 non-null  int16  
 3   PS_SUPPLYCOST  800000 non-null  float64
 4   PS_COMMENT     800000 non-null  object 
dtypes: float64(1), int16(2), int32(1), object(1)
memory usage: 150.0 MB


In [46]:
# Estadísticas descriptivas de la tabla partsupp
df_parts.describe(include = "all").T

,count,unique,top,freq,mean,std,min,25%,50%,75%,max
PS_PARTKEY,800000.0,NaN,NaN,NaN,100000.5,57735.063002,1.0,50000.75,100000.5,150000.25,200000.0
PS_SUPPKEY,800000.0,NaN,NaN,NaN,5000.5,2886.753136,1.0,2500.75,5000.5,7500.25,10000.0
PS_AVAILQTY,800000.0,NaN,NaN,NaN,5003.226934,2885.557591,1.0,2506.0,5003.0,7499.0,9999.0
PS_SUPPLYCOST,800000.0,NaN,NaN,NaN,500.525798,288.147954,1.0,250.81,500.41,750.2425,1000.0
PS_COMMENT,800000,799124,the fluffily unusual packages. slyly even dolp...,2,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [47]:
# Identificación de valores nulos
df_parts.isna().sum().sort_values(ascending = False)

PS_PARTKEY       0
PS_SUPPKEY       0
PS_AVAILQTY      0
PS_SUPPLYCOST    0
PS_COMMENT       0
dtype: int64

In [49]:
# Identificación de resgistros duplicados
df_ord.duplicated().sum()

np.int64(0)

In [51]:
# Identificación de clave única y de la duplicidad de producto-proveedor en algún registro
df_parts.groupby(["PS_PARTKEY", "PS_SUPPKEY"]).size().sum()

np.int64(800000)

###### REGION

In [52]:
reg= session.sql("SELECT * FROM TPCH_SF1.REGION")
reg.show()

#Creamos objeto pandas para poder explorar y visualizar los datos
df_reg = reg.to_pandas()
df_reg.head()

------------------------------------------------------------------------------------
|"R_REGIONKEY"  |"R_NAME"     |"R_COMMENT"                                         |
------------------------------------------------------------------------------------
|0              |AFRICA       |lar deposits. blithely final packages cajole. r...  |
|1              |AMERICA      |hs use ironic, even requests. s                     |
|2              |ASIA         |ges. thinly even pinto beans ca                     |
|3              |EUROPE       |ly final courts cajole furiously final excuse       |
|4              |MIDDLE EAST  |uickly special accounts cajole carefully blithe...  |
------------------------------------------------------------------------------------



,R_REGIONKEY,R_NAME,R_COMMENT
0,0,AFRICA,lar deposits. blithely final packages cajole. ...
1,1,AMERICA,"hs use ironic, even requests. s"
2,2,ASIA,ges. thinly even pinto beans ca
3,3,EUROPE,ly final courts cajole furiously final excuse
4,4,MIDDLE EAST,uickly special accounts cajole carefully blith...


In [ ]:
# Información genérica de la tabla region
df_reg.info(memory_usage='deep')

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 5 entries, 0 to 4
Data columns (total 3 columns):
 #   Column       Non-Null Count  Dtype 
---  ------       --------------  ----- 
 0   R_REGIONKEY  5 non-null      int8  
 1   R_NAME       5 non-null      object
 2   R_COMMENT    5 non-null      object
dtypes: int8(1), object(2)
memory usage: 1.0 KB


In [55]:
# Identificación de valores nulos
df_reg.isna().sum().sort_values(ascending = False)

R_REGIONKEY    0
R_NAME         0
R_COMMENT      0
dtype: int64

In [56]:
# Identificación de resgistros duplicados
df_ord.duplicated().sum()

np.int64(0)

In [57]:
# Identificación de productos únicos
df_reg.R_REGIONKEY.value_counts().sort_values(ascending = False)

R_REGIONKEY
0    1
1    1
2    1
3    1
4    1
Name: count, dtype: int64

###### SUPPLIER

In [58]:
sup= session.sql("SELECT * FROM TPCH_SF1.SUPPLIER")
sup.show()

#Creamos objeto pandas para poder explorar y visualizar los datos
df_sup = sup.to_pandas()
df_sup.head()

-------------------------------------------------------------------------------------------------------------------------------------------------------------------------------
|"S_SUPPKEY"  |"S_NAME"            |"S_ADDRESS"                          |"S_NATIONKEY"  |"S_PHONE"        |"S_ACCTBAL"  |"S_COMMENT"                                         |
-------------------------------------------------------------------------------------------------------------------------------------------------------------------------------
|1            |Supplier#000000001  | N kD4on9OM Ipw3,gf0JBoQDd7tgrzrddZ  |17             |27-918-335-1736  |5755.94      |each slyly above the careful                        |
|2            |Supplier#000000002  |89eJ5ksX3ImxJQBvxObC,                |5              |15-679-861-2259  |4032.68      | slyly bold instructions. idle dependen             |
|3            |Supplier#000000003  |q1,G3Pj6OjIuUYfUoH18BFTKP5aU9bEV3    |1              |11-383-516-1199  |4192.40     

,S_SUPPKEY,S_NAME,S_ADDRESS,S_NATIONKEY,S_PHONE,S_ACCTBAL,S_COMMENT
0,1,Supplier#000000001,"N kD4on9OM Ipw3,gf0JBoQDd7tgrzrddZ",17,27-918-335-1736,5755.94,each slyly above the careful
1,2,Supplier#000000002,"89eJ5ksX3ImxJQBvxObC,",5,15-679-861-2259,4032.68,slyly bold instructions. idle dependen
2,3,Supplier#000000003,"q1,G3Pj6OjIuUYfUoH18BFTKP5aU9bEV3",1,11-383-516-1199,4192.40,blithely silent requests after the express dep...
3,4,Supplier#000000004,Bk7ah4CK8SYQTepEmvMkkgMwg,15,25-843-787-7479,4641.08,riously even requests above the exp
4,5,Supplier#000000005,Gcdm2rJRzl5qlTVzc,11,21-151-690-3663,-283.84,. slyly regular pinto bea


In [59]:
# Información genérica de la tabla region
df_sup.info(memory_usage='deep')

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 10000 entries, 0 to 9999
Data columns (total 7 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   S_SUPPKEY    10000 non-null  int16  
 1   S_NAME       10000 non-null  object 
 2   S_ADDRESS    10000 non-null  object 
 3   S_NATIONKEY  10000 non-null  int8   
 4   S_PHONE      10000 non-null  object 
 5   S_ACCTBAL    10000 non-null  float64
 6   S_COMMENT    10000 non-null  object 
dtypes: float64(1), int16(1), int8(1), object(4)
memory usage: 3.4 MB


In [61]:
# Identificación de valores nulos
df_sup.isna().sum().sort_values(ascending = False)

S_SUPPKEY      0
S_NAME         0
S_ADDRESS      0
S_NATIONKEY    0
S_PHONE        0
S_ACCTBAL      0
S_COMMENT      0
dtype: int64

In [62]:
# Identificación de resgistros duplicados
df_sup.duplicated().sum()

np.int64(0)

In [64]:
# Identificación de proveeddores únicos
df_sup.S_SUPPKEY.value_counts().sort_values(ascending = False)

S_SUPPKEY
1985    1
1986    1
1987    1
1988    1
1989    1
       ..
1980    1
1981    1
1982    1
1983    1
1984    1
Name: count, Length: 10000, dtype: int64